# 08 · BatchNorm y augmentation
*Normalizar cada canal con el batch, los dos modos de la red, y ver más datos sin tener más*

Notebook del episodio 8 de **Visión: de convoluciones a ViT** ([ver la lección](https://d32vh5r3pvn9et.cloudfront.net/cursos/vision/batchnorm-y-augmentation)). Es el mismo guion del video, en texto, con cada bloque de código para correr y su salida real.

**Mecanismo:** BN a mano = nn.BatchNorm2d (7e-7), 4.096 valores por canal → ejemplo 4,7,5,2,7: μ = 5, σ² = 3,6 → pesos ×10: misma salida y gradiente ×0,1 → plana-8 sin/con BN: test en epoch 1 10 % vs. 40 % (lr 0,02), final 74,3 vs. 78,2 → running mean converge a 4,994; una imagen de valor 6: eval 1,0, train 0,0 → ResNet-20 del ep07: eval 84,2 %, train batch 500 84,4 %, batch 4 70,6 %, de a 1 11 % → paper BN: 14× menos pasos, 4,9 % top-5 → flip + recorte con padding 4 → ResNet-20 sin aug: train 100 % / test 84,2 % (brecha 15,8); con aug: 96,5 / 89,1 (brecha 7,4)

**Cómo correrlo:** con el entorno de `~/Movies/serie-vision/lab` (`cd lab && uv run --project ~/Movies/serie-alineacion/lab --group notebooks --with jupyterlab jupyter lab ../notebooks`, o en VS Code eligiendo el kernel `lab/.venv`). Todo corre offline: Gemma 2 2B sale de la caché de Hugging Face, y los entrenamientos son de juguete, con datos sintéticos.

Cada bloque de código arranca de cero (`%reset -f`): la celda de **preparación** es lo que el video no muestra, y la siguiente es exactamente el código que aparece en pantalla.

## ▶ Antes de empezar

Corré la celda de abajo una vez: baja el código de la serie y sus dependencias. Funciona en Google Colab (gratis) y en cualquier Jupyter. Para guardar tus cambios en Colab: **Archivo → Guardar una copia en Drive**.

In [ ]:
import os, subprocess, sys
SERIE = "vision"
REPO = "https://github.com/Faridmurzone/paperclase-notebooks"

def _raiz():  # carpeta del repo si ya estamos adentro
    d = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, ".paperclase-notebooks")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

RAIZ = _raiz()
if RAIZ is None:
    RAIZ = os.path.abspath("paperclase-notebooks")
    if not os.path.isdir(RAIZ):
        subprocess.run(["git", "clone", "--depth", "1", REPO, RAIZ], check=True)
os.chdir(os.path.join(RAIZ, SERIE, "lab"))  # los bloques de código importan desde snippets/ y leen datos/
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)
os.environ["HF_HUB_OFFLINE"] = "0"  # el notebook original corría sin red; acá baja los modelos de Hugging Face
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("listo:", os.getcwd())

In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import warnings; warnings.filterwarnings("ignore")

*Visión, de las convoluciones a los transformers. Episodio ocho: BatchNorm y data augmentation. En el episodio anterior usamos BatchNorm en todas las redes sin explicarla. Hoy la escribimos a mano, vemos por qué tiene dos modos y el bug que eso produce, y medimos qué pasa sin ella. Después, la otra mitad: cómo hacer que la red no memorice.*

### Dos herramientas, dos problemas

- BatchNorm: cada canal, a media 0 y varianza 1
- Con la estadística del batch al entrenar, y una fija al evaluar
- Overfitting: acertar el train y no el test
- Augmentation: la misma foto, espejada y corrida

BatchNorm toma la salida de una capa y la lleva, canal por canal, a media cero y varianza uno, y después la escala con dos parámetros que se aprenden. Para eso usa la media del batch durante el entrenamiento y una media guardada al evaluar: dos modos distintos. El otro problema es el overfitting: la red acierta casi todo el train y bastante menos el test, porque memorizó. Data augmentation lo ataca mostrando cada foto con pequeñas transformaciones que no cambian su clase.

## Parte 1 · BatchNorm a mano

Cuatro líneas de PyTorch, una por cada término de la fórmula.

### Normalizar por canal

$$
\mu_B = \tfrac{1}{m}\textstyle\sum_{i=1}^{m} x_i
$$
<small>m: todos los valores de ese canal en el batch</small>

$$
\sigma_B^2 = \tfrac{1}{m}\textstyle\sum_{i} (x_i - \mu_B)^2
$$
<small>varianza del batch: promedio de los desvíos al cuadrado</small>

$$
\hat x_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \varepsilon}}
$$
<small>ε = 0,00001 para no dividir por cero</small>

$$
y_i = \gamma\, \hat x_i + \beta
$$
<small>γ y β: un par por canal, se aprenden</small>

Con números primero: cinco valores de un canal, cuatro, siete, cinco, dos y siete. Mu sub B es su promedio, cinco. En la red, m son todos los valores de ese canal en el batch: imágenes por alto por ancho. Sigma cuadrado sub B es la varianza: restar la media, elevar al cuadrado y promediar. En el dibujo: uno, cuatro, cero, nueve y cuatro, que promedian tres coma seis. Restamos la media y dividimos por la raíz de la varianza; épsilon evita dividir por cero. Los valores quedan en media cero y varianza uno. Y al final, gamma y beta, dos números por canal que se aprenden: si a la red le conviene otra escala, la recupera.

### BatchNorm2d, escrita a mano

El tensor tiene forma batch por canales por alto por ancho. La media se toma sobre las dimensiones cero, dos y tres: todo menos el canal. Queda un número por canal. La varianza igual, dividiendo por m y no por m menos uno, como hace PyTorch al normalizar. Normalizamos y aplicamos gamma y beta, uno por canal. La entrada tiene media cinco y desvío tres; la salida, media cero y desvío uno. La diferencia con nn punto BatchNorm2d es de siete por diez a la menos siete. Y cada media se calcula sobre cuatro mil noventa y seis valores: sesenta y cuatro imágenes por ocho por ocho píxeles.

**Cada línea, su término:**

- `mu = x.mean(dim=(0, 2, 3), keepdim=True)` → $\mu_B$ — promedio sobre batch, alto y ancho
- `var = x.var(dim=(0, 2, 3), keepdim=True, unbiased=False)` → $\sigma_B^2$ — dividiendo por m, no por m − 1
- `x_hat = (x - mu) / torch.sqrt(var + eps)` → $\hat x = (x - \mu_B)/\sqrt{\sigma_B^2 + \varepsilon}$ — normalizar
- `return gamma.view(1, -1, 1, 1) * x_hat` → $y = \gamma \hat x + \beta$ — un γ y un β por canal

In [2]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep08_bn_mano.py
__file__ = 'snippets/ep08_bn_mano.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
torch.manual_seed(0)
x = torch.randn(64, 16, 8, 8) * 3 + 5        # batch de 64, 16 canales, 8×8; media 5, desvío 3

In [3]:
def batchnorm(x, gamma, beta, eps=1e-5):
    mu = x.mean(dim=(0, 2, 3), keepdim=True)            # μ_B: una por canal
    var = x.var(dim=(0, 2, 3), keepdim=True, unbiased=False)
    x_hat = (x - mu) / torch.sqrt(var + eps)             # media 0, var 1
    return gamma.view(1, -1, 1, 1) * x_hat + beta.view(1, -1, 1, 1)

bn = torch.nn.BatchNorm2d(16)                # γ = 1, β = 0 al inicio
y = batchnorm(x, bn.weight, bn.bias)
r = lambda t: f"{abs(t.mean()):.2f} / {t.std():.2f}"
print("media / desvío antes:  ", r(x))
print("media / desvío después:", r(y))
print("dif. con nn.BatchNorm2d:", f"{(y - bn(x)).abs().max():.1e}")
print("valores por canal:", 64 * 8 * 8)

media / desvío antes:   4.99 / 3.00
media / desvío después: 0.00 / 1.00
dif. con nn.BatchNorm2d: 7.2e-07
valores por canal: 4096


### Precisión de test por epoch, red plana de 8 capas

Para medir qué aporta, entrenamos la red plana de ocho capas del episodio anterior cuatro veces, treinta epochs cada una. Sin BatchNorm y learning rate cero coma cero dos: arranca muy lento. Con BatchNorm, mismo learning rate: en la primera epoch ya pasa el cuarenta por ciento. Sin BatchNorm y un learning rate cinco veces más grande: mejora, pero sigue atrás al principio. Y con BatchNorm y el learning rate alto: la mejor curva. La diferencia grande está en las primeras epochs.

### Las cuatro corridas

Los números de esas cuatro corridas, leídos del registro del entrenamiento. Un diccionario con las cuatro historias y la precisión de test en la epoch uno, la cinco y la treinta. Sin BatchNorm, después de una epoch la red sigue en diez por ciento: adivina al azar entre diez clases. Con BatchNorm, cuarenta. En la epoch cinco, veintisiete contra sesenta y ocho al learning rate chico. Al final de treinta epochs las distancias se achican: setenta y cuatro contra setenta y ocho. A esta escala, BatchNorm acelera mucho el arranque y gana unos pocos puntos al final.

In [4]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep08_velocidad.py
__file__ = 'snippets/ep08_velocidad.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os, json
AQUI = os.path.dirname(os.path.abspath(__file__))
h7 = json.load(open(f"{AQUI}/../datos/ep07_hist.json"))
h8 = json.load(open(f"{AQUI}/../datos/ep08_hist.json"))

In [5]:
corridas = {"sin BN, lr 0,02": h8["sinbn_lr002"], "con BN, lr 0,02": h8["conbn_lr002"],
            "sin BN, lr 0,1": h8["sinbn_lr01"], "con BN, lr 0,1": h7["plana8"]}
print("red plana de 8 capas   test ep. 1   ep. 5   ep. 30")
for nombre, c in corridas.items():
    t = [e["acc_test"] for e in c["hist"]]
    print(f"{nombre:22} {t[0]:8.1%} {t[4]:7.1%} {t[-1]:7.1%}")

red plana de 8 capas   test ep. 1   ep. 5   ep. 30
sin BN, lr 0,02           10.0%   27.2%   74.3%
con BN, lr 0,02           39.9%   68.4%   78.2%
sin BN, lr 0,1            10.0%   48.5%   77.7%
con BN, lr 0,1            43.8%   68.9%   79.8%


### Por qué la escala de los pesos deja de importar

$$
z = a\,Wx \;\Rightarrow\; \mu_B \to a\,\mu_B,\quad \sigma_B \to |a|\,\sigma_B
$$
<small>a: un escalar que multiplica todos los pesos de la capa</small>

$$
\frac{a\,z - a\,\mu_B}{|a|\,\sigma_B} = \frac{z - \mu_B}{\sigma_B}\quad (a > 0)
$$
<small>la a se cancela: la salida no cambia</small>

$$
\frac{\partial L}{\partial (aW)} = \frac{1}{a}\,\frac{\partial L}{\partial W}
$$
<small>pesos más grandes ⇒ gradiente más chico: se autorregula</small>

Una propiedad que explica parte de la aceleración. Si multiplicamos todos los pesos de una capa por un número a, su salida z se multiplica por a, y con ella la media y el desvío del batch. En el dibujo, los cinco valores de antes, diez veces más grandes: la media pasa de cinco a cincuenta. Al normalizar, a aparece arriba y abajo y se cancela. En el dibujo, los mismos valores normalizados de antes: menos cero coma cincuenta y tres, uno coma cero cinco, y así. Y como la salida no cambia, el gradiente respecto de los pesos escalados es uno sobre a por el original: si los pesos crecen, los pasos se achican solos. Por eso la red tolera learning rates más altos.

### Pesos ×10, misma salida

Lo comprobamos. Una convolución, y los mismos pesos multiplicados por una escala. La salida pasa por una BatchNorm en modo train. Corremos con escala uno y con escala diez, con la misma pérdida. Y comparamos los gradientes. Antes de la normalización, el desvío pasa de cero coma cinco cuatro a cinco coma cuatro uno: diez veces. Después, la diferencia máxima es uno por diez a la menos cuatro, que viene solo de épsilon. Y el gradiente con pesos diez veces más grandes es exactamente un décimo.

In [6]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep08_escala.py
__file__ = 'snippets/ep08_escala.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
torch.manual_seed(0)
x = torch.randn(64, 3, 8, 8)
objetivo = torch.randn(64, 16, 8, 8)               # una pérdida cualquiera, fija

In [7]:
conv = torch.nn.Conv2d(3, 16, 3, padding=1, bias=False)
bn = torch.nn.BatchNorm2d(16)                # modo train: media del batch

def paso(escala):
    w = (conv.weight.detach() * escala).requires_grad_(True)
    z = torch.nn.functional.conv2d(x, w, padding=1)
    y = bn(z)
    ((y - objetivo) ** 2).mean().backward()
    return z.std().item(), y, w.grad.norm().item()

s1, y1, g1 = paso(1.0)
s10, y10, g10 = paso(10.0)                   # pesos 10 veces más grandes
print("desvío antes de la BN:", round(s1, 2), "→", round(s10, 2))
print("después de la BN, dif. máx.:", f"{(y10 - y1).abs().max():.1e}")
print("gradiente con pesos ×10 / ×1:", round(g10 / g1, 3))

desvío antes de la BN: 0.54 → 5.41
después de la BN, dif. máx.: 1.1e-04
gradiente con pesos ×10 / ×1: 0.1


## Parte 2 · Modo train y modo eval

Al evaluar puede haber una sola imagen. ¿De dónde sale la media del batch?

### La media que se guarda

$$
\hat\mu \leftarrow (1 - 0{,}1)\, \hat\mu + 0{,}1\, \mu_B
$$
<small>en cada paso de entrenamiento: running mean</small>

$$
\text{eval: } y = \gamma\, \frac{x - \hat\mu}{\sqrt{\hat\sigma^2 + \varepsilon}} + \beta
$$
<small>con lo guardado: no depende de las otras imágenes</small>

Durante el entrenamiento, además de normalizar, BatchNorm actualiza una media guardada, mu sombrero, que en PyTorch se llama running mean: se queda con noventa por ciento de lo que tenía y le suma diez por ciento de la media del batch nuevo. Es un promedio móvil exponencial: las medias viejas pesan cada vez menos. En el dibujo, las dos columnas con sus pesos. Lo mismo para la varianza. Al evaluar, se usa lo guardado. Así la salida de una imagen no depende de qué otras imágenes vinieron en el batch, y funciona con una sola.

### Una imagen, dos modos

Doscientos batches con media cinco. La media guardada converge a cuatro coma nueve nueve, y la varianza a cero coma nueve nueve. Ahora una sola imagen, con todos los valores en seis. En modo eval, la normalización usa la media guardada: seis menos cinco, sobre uno, da uno. En modo train, la media del batch es la de esa imagen sola, seis, y la salida es cero: la imagen se borró. Un detalle más: en modo train, cada llamada también actualiza la media guardada.

In [8]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep08_running.py
__file__ = 'snippets/ep08_running.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
torch.manual_seed(0)
bn = torch.nn.BatchNorm2d(1)

In [9]:
for paso in range(200):                      # entrenamiento: batches de media 5
    bn(torch.randn(32, 1, 4, 4) + 5)
print("running_mean:", round(bn.running_mean.item(), 3),
      " running_var:", round(bn.running_var.item(), 3))

una = torch.full((1, 1, 4, 4), 6.0)          # UNA imagen, valor 6
bn.eval()                                    # usa running_mean/var
print("eval:  ", round(bn(una).mean().item(), 3))
bn.train()                                   # usa la media de ESTE batch
print("train: ", round(bn(una).mean().item(), 3))

running_mean: 4.994  running_var: 0.994
eval:   1.009
train:  0.0


### El bug clásico: evaluar en modo train

Ahora con la ResNet de veinte capas del episodio siete, ya entrenada, en dos mil imágenes de test. La función cambia el modo de la red y evalúa con un tamaño de batch dado. Modo eval, de a una imagen: ochenta y cuatro coma dos por ciento. Modo train con batches de quinientas: ochenta y cuatro coma cuatro, casi igual, porque la media de quinientas imágenes se parece a la guardada. Con batches de cuatro baja a setenta coma seis. Y de a una imagen, once por ciento: azar. Es el bug clásico de olvidarse model punto eval: con batches grandes no se nota, y en producción, de a una imagen, la red se rompe.

In [10]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep08_modos.py
__file__ = 'snippets/ep08_modos.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os, sys
import torch
AQUI = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, AQUI)
from _cifar import cargar, normalizar
from _resnet import cargar_red
torch.set_grad_enabled(False)
X, y = cargar("test", n=2000)
X = normalizar(X)
red, _ = cargar_red(f"{AQUI}/../datos/ep07_res20.pt")   # la ResNet-20 del ep07

In [11]:
def precision(red, modo_train, batch):
    red.train(modo_train)                    # cambia qué estadística usa la BN
    pred = torch.cat([red(X[i:i + batch]).argmax(1)
                      for i in range(0, len(X), batch)])
    return (pred == y).float().mean().item()

print("eval,  batch 1:   ", f"{precision(red, False, 1):.1%}")
print("train, batch 500: ", f"{precision(red, True, 500):.1%}")
print("train, batch 4:   ", f"{precision(red, True, 4):.1%}")
print("train, batch 1:   ", f"{precision(red, True, 1):.1%}")

eval,  batch 1:    84.2%


train, batch 500:  84.4%


train, batch 4:    70.6%


train, batch 1:    11.0%


### La misma red, evaluada en cuatro modos

Las cuatro mediciones, como barras. En modo eval da lo mismo el tamaño del batch: ochenta y cuatro coma dos. En modo train con quinientas imágenes casi no se nota. Con cuatro, pierde catorce puntos. Y con una sola, cae al azar. Si alguna vez un modelo anda bien en el notebook y mal en producción, esta es una de las primeras cosas a revisar.

- **eval · de a 1**: 84,2 %
- **train · batch 500**: 84,4 %
- **train · batch 4**: 70,6 %
- **train · de a 1**: 11,0 %

> **Ioffe y Szegedy, 2015 — Batch Normalization** · [arXiv:1502.03167](https://arxiv.org/abs/1502.03167)
>
> **14× menos pasos** · «Batch Normalization achieves the same accuracy with 14 times fewer training steps»; con un ensemble, 4,9 % de error top-5 en validación de ImageNet.

El paper original. Aplicada a un clasificador de ImageNet de ese momento, la BatchNorm alcanza la misma precisión con catorce veces menos pasos de entrenamiento, y con un ensemble de redes normalizadas llegaron a cuatro coma nueve por ciento de error top-5. Ellos lo explicaron con el internal covariate shift: que la distribución de entrada de cada capa cambia mientras se entrenan las anteriores.

## Parte 3 · Overfitting y augmentation

Treinta epochs sobre las mismas cincuenta mil fotos: la red termina sabiéndolas de memoria.

### Espejo y recorte

Dos transformaciones, las que usa el paper de ResNet en CIFAR-10, del Canadian Institute for Advanced Research. Con probabilidad un medio, espejar horizontalmente: un auto mirando a la izquierda sigue siendo un auto. Agregar un borde de cuatro píxeles de ceros, que deja la imagen en cuarenta por cuarenta, elegir una esquina al azar y recortar treinta y dos por treinta y dos. El resultado es la misma foto corrida hasta cuatro píxeles en cada dirección. Tres versiones del mismo auto: dos sin espejo y corridas distinto, y una espejada. En el entrenamiento, cada epoch ve una versión nueva de cada foto.

In [12]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep08_aug.py
__file__ = 'snippets/ep08_aug.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os, sys, json
import torch
import torch.nn.functional as F
AQUI = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, AQUI)
from _cifar import cargar
X, y = cargar("test", n=200)
i = int((y == 1).nonzero()[0])                     # el primer auto del test
img = X[i:i + 1].float() / 255                     # (1, 3, 32, 32), sin normalizar para mirarla
torch.manual_seed(1)

In [13]:
def augmentar(x, pad=4):
    espejo = bool(torch.rand(1) < 0.5)
    if espejo:
        x = x.flip(3)                              # espejo horizontal
    xp = F.pad(x, (pad, pad, pad, pad))            # 40×40, borde de ceros
    dy, dx = torch.randint(0, 2 * pad + 1, (2,)).tolist()
    return xp[:, :, dy:dy + 32, dx:dx + 32], espejo, (dy - pad, dx - pad)

versiones = []
for k in range(3):
    v, espejo, (dy, dx) = augmentar(img)
    versiones.append(v)
    print(f"versión {k + 1}: {tuple(v.shape)}  espejo={espejo}"
          f"  corrida ({dy:+d}, {dx:+d})")

versión 1: (1, 3, 32, 32)  espejo=False  corrida (+1, -4)
versión 2: (1, 3, 32, 32)  espejo=False  corrida (+3, -3)
versión 3: (1, 3, 32, 32)  espejo=True  corrida (+1, +4)


In [14]:
# Verificación extra (corrió junto con el video)
gris = lambda t: (0.299 * t[0, 0] + 0.587 * t[0, 1] + 0.114 * t[0, 2])
paneles = [gris(img), gris(versiones[0]), gris(versiones[2])]
filas = [sum(([round(float(p[r, c]), 2) for c in range(32)] + ([0.0] if j < len(paneles) - 1 else [])
              for j, p in enumerate(paneles)), []) for r in range(32)]
json.dump({"filas": filas}, open(os.path.join(AQUI, "..", "datos", "ep08_aug_fig.json"), "w"))

### El mismo auto, tres veces

A la izquierda, el original, en escala de grises. En el medio, la versión uno: corrida hacia la derecha y apenas hacia arriba, con una franja negra de padding a la izquierda. A la derecha, la versión tres: espejada y corrida hacia el otro lado. Para nosotros es obviamente el mismo auto; para la red son tres tensores distintos, y tiene que aprender que los tres son auto.

### Precisión de train y test, ResNet-20

Ya en el episodio cinco la red convolucional chica tenía ochenta y ocho coma nueve por ciento en train y setenta y cuatro coma siete en test. Volvemos a la ResNet de veinte capas, mirando las dos curvas. Sin augmentation, la precisión de train llega a cien por ciento: y la de test se queda bastante más abajo. Esa distancia es el overfitting. Con augmentation, el train ya no llega a cien: cada epoch ve versiones nuevas y no puede memorizarlas. Y el test sube: desde la mitad del entrenamiento queda por encima de la curva sin augmentation.

### La brecha, medida

Las dos ResNet de veinte capas, la entrenada sin augmentation y la entrenada con augmentation, mismas treinta epochs. Medimos sobre dos mil imágenes que vieron en el entrenamiento y dos mil que nunca vieron. Sin augmentation: cien por ciento en train, ochenta y cuatro coma dos en test, una brecha de quince coma ocho puntos. Con augmentation: noventa y seis coma cinco en train, ochenta y nueve coma uno en test. La brecha baja a siete coma cuatro, y el test sube casi cinco puntos, sin una sola imagen nueva.

In [15]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep08_overfit.py
__file__ = 'snippets/ep08_overfit.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os, sys
import torch
AQUI = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, AQUI)
from _cifar import cargar, normalizar
from _resnet import cargar_red
from _redes import evaluar
torch.set_grad_enabled(False)
Xtr, ytr = cargar("train", n=2000); Xte, yte = cargar("test", n=2000)
Xtr, Xte = normalizar(Xtr), normalizar(Xte)

In [16]:
for nombre, ruta in [("sin augmentation", "ep07_res20.pt"),
                     ("con augmentation", "ep08_res20_aug.pt")]:
    red, _ = cargar_red(f"{AQUI}/../datos/{ruta}")   # ResNet-20, 30 epochs
    tr = evaluar(red, Xtr, ytr)              # 2.000 imágenes que vio
    te = evaluar(red, Xte, yte)              # 2.000 que nunca vio
    print(f"{nombre}: train {tr:.1%}  test {te:.1%}  brecha {tr - te:.1%}")

sin augmentation: train 100.0%  test 84.2%  brecha 15.8%


con augmentation: train 96.5%  test 89.1%  brecha 7.4%


## Parte 4 · Reglas prácticas

Dónde va cada cosa, y los errores que más se repiten.

### Cuándo usar cada una

- BatchNorm: después de cada conv, antes de la ReLU
- Batches chicos: la media del batch es ruidosa
- model.eval() antes de evaluar, siempre
- Augmentation: solo transformaciones que no cambian la clase

Cuatro reglas prácticas. BatchNorm va después de cada convolución y antes de la ReLU, rectified linear unit; por eso esas convoluciones no llevan sesgo: beta ya cumple ese papel. Con batches muy chicos, la media del batch es ruidosa, y BatchNorm funciona peor. Antes de evaluar, siempre model punto eval, y model punto train al volver a entrenar. Y la augmentation solo con transformaciones que no cambian la clase: espejar un auto está bien; rotar ciento ochenta grados un seis lo convierte en un nueve.

## Resumiendo

- BatchNorm: (x − μ_B)/√(σ²_B + ε), después γ y β, por canal
- Train: estadística del batch; eval: la running mean guardada
- Bug: modo train de a 1 imagen → 11 % (eval: 84,2 %)
- Sin BN, epoch 1 en 10 %; con BN, 40 %
- Augmentation: espejo y recorte con padding de 4
- Próximo: cortar la imagen en patches, el Vision Transformer